# 画像の説明文を評価する（BLEU・CIDEr・CLIPScore）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/30_vision_language/caption_eval.ipynb)

NLP クックブック 30-03 ／ 画像と言語

画像の説明文（キャプション）を、参照のキャプションとの一致（BLEU・CIDEr）と、画像そのものとの一致 (CLIPScore) で自動評価したい。それぞれの指標がどんな誤りに強く、どんな誤りに弱いかを、日本語のキャプションで確かめたい。この章の「視覚言語モデルで画像を説明し、画像について質問に答える (VQA)」で作った説明文も、この方法で評価できる。

In [1]:
# pycocoevalcap は METEOR や SPICE の Java の部品も含むので、ダウンロードが約 100MB ある (ここでは CIDEr だけを使う)
!pip install -q "sacrebleu[ja]" pycocoevalcap

In [2]:
# STAIR Captions: MS COCO の画像 1 枚に日本語のキャプションを 5 つ付けたデータ (キャプションは CC BY 4.0)
# 画像は COCO (Flickr) の写真で、データセットのライセンスとは別に 1 枚ごとにライセンスが違う (license 列。非営利のみ・改変禁止のものも含む)
# ここでは画像を表示せず、評価に使うだけにする
# 検証セットの最初のファイルから 300 枚 (100 枚ずつの行グループ 3 つ、約 50MB) だけを読む
import io
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image

path = "datasets/shunk031/STAIR-Captions/v1.2.0/validation-00000-of-00014.parquet"
with HfFileSystem().open(path, block_size=8 << 20) as f:
    rows = pq.ParquetFile(f).read_row_groups([0, 1, 2], columns=["image_id", "image", "annotations"]).to_pylist()

images = [Image.open(io.BytesIO(r["image"]["bytes"])).convert("RGB") for r in rows]
captions = [r["annotations"]["caption"] for r in rows]
print(len(images), "枚、キャプションの数:", {len(c) for c in captions})
for c in captions[0]:
    print(" ", c)

300 枚、キャプションの数: {5}
  赤いヘルメットをかぶった人がバイクに跨っている
  赤い服の男性がバイクに乗っている
  赤いシャツを着た男性がバイクに乗っている
  赤いシャツを着た男性がバイクに跨っている
  赤い服を着た人がバイクに乗っている


In [3]:
# 1 つ目のキャプションを「候補」(システムの出力の代わり)、残り 4 つを参照にする
import random

cand = [c[0] for c in captions]
refs = [c[1:] for c in captions]

# わざと誤りを入れた候補を作る: 種類ごとの置き換えの表。文中で最初に見つかった 1 か所だけを置き換える
EDITS = {
    "人物": {"男性": "女性", "女性": "男性", "男の子": "女の子", "女の子": "男の子", "少年": "少女", "少女": "少年"},
    "物": {"犬": "猫", "猫": "犬", "牛": "馬", "馬": "牛", "電車": "バス", "バス": "電車", "自転車": "バイク", "バイク": "自転車",
          "テーブル": "ベッド", "ベッド": "テーブル", "ピザ": "ケーキ", "ケーキ": "ピザ", "キッチン": "浴室", "トイレ": "冷蔵庫",
          "キリン": "ゾウ", "象": "キリン", "ゾウ": "キリン", "飛行機": "船", "皿": "箱"},
    "色": {"赤い": "青い", "青い": "赤い", "白い": "黒い", "黒い": "白い", "白と黒": "赤と青", "茶色": "緑色", "緑": "赤"},
    "位置・数": {"の上に": "の下に", "の下に": "の上に", "の前": "の後ろ", "の横": "の上", "左": "右", "右": "左",
               "一人": "三人", "二人": "一人", "一頭": "三頭", "二頭": "一頭", "1人": "3人", "2人": "1人", "たくさんの": "一つの"},
    "言い換え (意味は同じ)": {"男性": "男の人", "女性": "女の人", "少年": "男の子", "少女": "女の子", "子供": "子ども",
                      "パソコン": "コンピューター", "バイク": "オートバイ", "道路": "道", "机": "テーブル", "沢山": "たくさん"},
}

def replace_first(s, table):
    hits = [(s.find(k), -len(k), k) for k in table if k in s]
    if not hits:
        return None
    i, _, k = min(hits)
    return s[:i] + table[k] + s[i + len(k):]

variants = {kind: [replace_first(s, table) for s in cand] for kind, table in EDITS.items()}
rng = random.Random(0)
perm = list(range(len(cand)))
rng.shuffle(perm)
variants["別の画像"] = [cand[j] if j != i else cand[(i + 1) % len(cand)] for i, j in enumerate(perm)]   # 別の画像のキャプション
other = variants["別の画像"]

for kind, vs in variants.items():
    i = next(i for i, v in enumerate(vs) if v is not None)
    print(f"{kind} ({sum(v is not None for v in vs)} 枚): {cand[i]} → {vs[i]}")

人物 (39 枚): 赤い服を着た女性がケーキをカットしている → 赤い服を着た男性がケーキをカットしている
物 (123 枚): 赤いヘルメットをかぶった人がバイクに跨っている → 赤いヘルメットをかぶった人が自転車に跨っている
色 (63 枚): 赤いヘルメットをかぶった人がバイクに跨っている → 青いヘルメットをかぶった人がバイクに跨っている
位置・数 (61 枚): キッチンにエプロンをした男性が2人いる → キッチンにエプロンをした男性が1人いる
言い換え (意味は同じ) (71 枚): 赤いヘルメットをかぶった人がバイクに跨っている → 赤いヘルメットをかぶった人がオートバイに跨っている
別の画像 (300 枚): 赤いヘルメットをかぶった人がバイクに跨っている → 石造りの教会の前に馬が3頭草を食べている


In [4]:
# BLEU と CIDEr は、日本語を単語に分割してから n-gram を数える。sacreBLEU の ja-mecab と同じ分割を CIDEr にも使う
from sacrebleu.metrics import BLEU
from sacrebleu.tokenizers.tokenizer_ja_mecab import TokenizerJaMecab
from pycocoevalcap.cider.cider import Cider   # CIDEr-D (n=1〜4、長さの罰則と頻度の上限つき、10 倍した値)

mecab = TokenizerJaMecab()
print(mecab(cand[0]))

def bleu_corpus(hyps, refsets, tokenize="ja-mecab"):
    # 参照は「参照セットのリスト」に組み替える。zip は、画像ごとに参照の数が違うと最も少ない数に切りそろえてしまうので注意
    return BLEU(tokenize=tokenize).corpus_score(hyps, list(map(list, zip(*refsets)))).score

sent_bleu = BLEU(tokenize="ja-mecab", effective_order=True)
def bleu_sent(hyps, refsets):
    return [sent_bleu.sentence_score(h, r).score for h, r in zip(hyps, refsets)]

def cider(hyps, refsets, tok=mecab):
    gts = {i: [tok(r) for r in rs] for i, rs in enumerate(refsets)}
    res = {i: [tok(h)] for i, h in enumerate(hyps)}
    mean, per_image = Cider().compute_score(gts, res)
    return mean, per_image

print(f"人のキャプション (参照 4 つ): BLEU = {bleu_corpus(cand, refs):.1f}  CIDEr = {cider(cand, refs)[0]:.3f}")
refs2 = [r[:2] for r in refs]
print(f"人のキャプション (参照 2 つ): BLEU = {bleu_corpus(cand, refs2):.1f}  CIDEr = {cider(cand, refs2)[0]:.3f}")

赤い ヘルメット を かぶっ た 人 が バイク に 跨っ て いる
人のキャプション (参照 4 つ): BLEU = 25.3  CIDEr = 0.806
人のキャプション (参照 2 つ): BLEU = 17.1  CIDEr = 0.822


In [5]:
# 単語に分割しないとどうなるか。13a (英語向け) では日本語の文全体がほぼ 1 トークンになる
for name, tok in [("ja-mecab", "ja-mecab"), ("char", "char"), ("13a", "13a")]:
    print(f"BLEU ({name:8}) 人 = {bleu_corpus(cand, refs, tok):5.1f}  別の画像 = {bleu_corpus(other, refs, tok):5.1f}")
for name, tok in [("MeCab", mecab), ("文字", lambda s: " ".join(s)), ("分割なし", lambda s: s)]:
    print(f"CIDEr ({name:4}) 人 = {cider(cand, refs, tok)[0]:.3f}  別の画像 = {cider(other, refs, tok)[0]:.3f}")

BLEU (ja-mecab) 人 =  25.3  別の画像 =   3.2
BLEU (char    ) 人 =  39.2  別の画像 =   9.3
BLEU (13a     ) 人 =   0.0  別の画像 =   0.0
CIDEr (MeCab) 人 = 0.806  別の画像 = 0.032
CIDEr (文字  ) 人 = 1.108  別の画像 = 0.073
CIDEr (分割なし) 人 = 0.002  別の画像 = 0.000


In [6]:
# CLIPScore (Hessel et al., 2021): 画像とキャプションの埋め込みのコサイン類似度 c から 2.5 × max(c, 0)。参照が要らない
# (元の論文は英語の CLIP で、文の前に "A photo depicts" を付けている。ここでは日本語の CLIP に文をそのまま入れる)
# RefCLIPScore: CLIPScore と「候補と参照の埋め込みの類似度の最大値」の調和平均。参照も使う
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoConfig, AutoImageProcessor, AutoTokenizer
from transformers.dynamic_module_utils import get_class_from_dynamic_module

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "line-corporation/clip-japanese-base"   # LY Corporation の日本語 CLIP (Apache-2.0)。この章の CLIP のレシピと同じモデルと読み込み方
rev = "77a62f8af977acd73ec2f927fb73fdaeb13af7d1"      # リポジトリのコミットを固定する (中のコードを実行するので、すべての読み込みに渡す)
clip_tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True, revision=rev)
clip_proc = AutoImageProcessor.from_pretrained(name, trust_remote_code=True, revision=rev)
# モデルカードの AutoModel.from_pretrained(name, trust_remote_code=True) は transformers 5 系ではエラーになるので、
# リポジトリ内のクラスを取り出して重みを直接読み込む
CLYPModel = get_class_from_dynamic_module("modeling_clyp.CLYPModel", name, revision=rev)  # modeling_clyp.py を取得してクラスを返す
clip = CLYPModel(AutoConfig.from_pretrained(name, trust_remote_code=True, revision=rev))  # config から重みが乱数のモデルを作る
print(clip.load_state_dict(load_file(hf_hub_download(name, "model.safetensors", revision=rev))))  # All keys matched と出れば全部の重みが入った
clip = clip.to(device).eval()                        # 重みは float32 のまま使う (約 0.8GB で、T4 にも CPU にも載る)

@torch.no_grad()
def embed_images(imgs, bs=32):
    out = [clip.get_image_features(**clip_proc(imgs[i:i + bs], return_tensors="pt").to(device)) for i in range(0, len(imgs), bs)]
    return torch.nn.functional.normalize(torch.cat(out).float(), dim=-1).cpu().numpy()

@torch.no_grad()
def embed_texts(texts, bs=64):
    out = [clip.get_text_features(**clip_tok(texts[i:i + bs]).to(device)) for i in range(0, len(texts), bs)]
    return torch.nn.functional.normalize(torch.cat(out).float(), dim=-1).cpu().numpy()

img_emb = embed_images(images)   # CPU (2 コア) では、モデルの読み込みを含めて 2〜4 分程度
ref_emb = [embed_texts(rs) for rs in refs]

def clip_score(hyps, idx=None):
    """idx: 各候補に対応する画像の番号 (省略時は 0, 1, 2, ...)"""
    idx = list(range(len(hyps))) if idx is None else idx
    return 2.5 * np.maximum((embed_texts(hyps) * img_emb[idx]).sum(-1), 0)

def ref_clip_score(hyps, idx=None):
    idx = list(range(len(hyps))) if idx is None else idx
    h = embed_texts(hyps)
    cs = 2.5 * np.maximum((h * img_emb[idx]).sum(-1), 0)
    rs = np.array([max(0, (ref_emb[i] @ v).max()) for i, v in zip(idx, h)])
    return 2 * cs * rs / np.maximum(cs + rs, 1e-8)   # 両方 0 のときの 0 除算を避ける

print(f"CLIPScore 人のキャプション = {clip_score(cand).mean():.3f}  別の画像 = {clip_score(other).mean():.3f}")

<All keys matched successfully>
CLIPScore 人のキャプション = 0.899  別の画像 = 0.250


In [7]:
# 誤りの種類ごとに、置き換えができた画像だけを集めて、元の候補と誤りを入れた候補を比べる
import pandas as pd

def evaluate(hyps, idx):
    h, r = [hyps[i] for i in idx], [refs[i] for i in idx]
    each = dict(文BLEU=np.array(bleu_sent(h, r)), CIDEr=cider(h, r)[1], CLIPScore=clip_score(h, idx), RefCLIPScore=ref_clip_score(h, idx))
    return {"BLEU": bleu_corpus(h, r), **{k: v.mean() for k, v in each.items()}}, each

ratio, wins = [], []
for kind, hyps in variants.items():
    idx = [i for i, s in enumerate(hyps) if s is not None]
    base, base_each = evaluate(cand, idx)     # 同じ画像の集合で、元の (人の) 候補も評価する
    test, test_each = evaluate(hyps, idx)
    ratio.append({"種類": kind, "枚数": len(idx), **{k: f"{test[k] / base[k]:.0%}" for k in base}})
    # 1 枚ごとに「元の候補のほうが高い」割合。意味が変わる誤りなら 100% に近いほど、誤りをよく見分けている
    # 値が変わらない (同点) 場合は括弧内に割合を示す。置き換えた語が参照にも出てこなければ、BLEU は変わらないことが多い
    wins.append({"種類": kind, **{k: f"{(base_each[k] > test_each[k]).mean():.0%} ({(np.isclose(base_each[k], test_each[k])).mean():.0%})"
                                for k in base_each}})
print("誤りを入れた候補の値 ÷ 元の候補の値")
display(pd.DataFrame(ratio))
print("1 枚ごとに、元の候補のほうが高く評価された割合 (括弧内は同点の割合)")
display(pd.DataFrame(wins))

誤りを入れた候補の値 ÷ 元の候補の値


,種類,枚数,BLEU,文BLEU,CIDEr,CLIPScore,RefCLIPScore
0,人物,39,83%,86%,92%,92%,91%
1,物,123,76%,79%,68%,83%,85%
2,色,63,90%,92%,91%,89%,89%
3,位置・数,61,71%,76%,81%,97%,96%
4,言い換え (意味は同じ),71,86%,89%,88%,101%,100%
5,別の画像,300,13%,32%,4%,28%,33%


1 枚ごとに、元の候補のほうが高く評価された割合 (括弧内は同点の割合)


,種類,文BLEU,CIDEr,CLIPScore,RefCLIPScore
0,人物,85% (13%),95% (0%),82% (0%),92% (0%)
1,物,89% (11%),95% (0%),92% (0%),98% (0%)
2,色,51% (48%),83% (0%),90% (0%),89% (0%)
3,位置・数,66% (31%),87% (0%),70% (0%),80% (0%)
4,言い換え (意味は同じ),70% (8%),73% (0%),48% (0%),55% (0%)
5,別の画像,90% (0%),98% (0%),99% (0%),100% (0%)


In [8]:
# CIDEr は参照の中でよく出る n-gram を IDF で割り引く。参照から求めた IDF の例
from collections import Counter

df = Counter(g for rs in refs for g in {w for r in rs for w in mecab(r).split()})
N = len(refs)
for w in ["が", "て", "いる", "の", "男性", "テーブル", "キリン", "パソコン"]:
    print(f"{w:8} 出てくる画像の数 = {df[w]:3}  IDF = {np.log(N / max(1, df[w])):.2f}")

# どの画像にも当てはまりそうな「ありがちな」候補を全画像に出したとき
for generic in ["男性が立っている", "テーブルの上に置いてある"]:
    g = [generic] * len(cand)
    print(f"{generic}: BLEU = {bleu_corpus(g, refs):.1f}  CIDEr = {cider(g, refs)[0]:.3f}  CLIPScore = {clip_score(g).mean():.3f}")

が        出てくる画像の数 = 299  IDF = 0.00
て        出てくる画像の数 = 288  IDF = 0.04
いる       出てくる画像の数 = 291  IDF = 0.03
の        出てくる画像の数 = 298  IDF = 0.01
男性       出てくる画像の数 =  41  IDF = 1.99
テーブル     出てくる画像の数 =  12  IDF = 3.22
キリン      出てくる画像の数 =  23  IDF = 2.57
パソコン     出てくる画像の数 =   7  IDF = 3.76
男性が立っている: BLEU = 5.9  CIDEr = 0.036  CLIPScore = 0.413
テーブルの上に置いてある: BLEU = 4.9  CIDEr = 0.046  CLIPScore = 0.389


- 人が書いたキャプションを同じ画像の別の人のキャプションで評価しても、BLEU は 25.3、CIDEr は 0.806 にとどまった（300 枚、参照 4 つ）。システムの値は、この「人同士の一致」を目安に読む。参照を 2 つに減らすと BLEU は 17.1 に下がった（どれかの参照と一致すればよいので、参照が多いほど一致しやすい）が、CIDEr は参照ごとの類似度の平均なので 0.822 とほとんど変わらなかった。参照の数が違う結果同士は比べない。
- 日本語は単語に分割してから数える。英語向けの `13a` では BLEU が 0.0、分割しない CIDEr は 0.002 になった。文字単位にすると BLEU 39.2・CIDEr 1.108 と MeCab で分割したとき（25.3・0.806）より高く出るので、分割の方法が違う値も比べられない（機械翻訳の章の「機械翻訳を BLEU で評価する (sacreBLEU)」の署名を参照）。pycocoevalcap の分割器 (PTBTokenizer) は英語用で Java も要るので使わず、MeCab で分割した文字列を `Cider` に渡している。
- n-gram の一致の指標は言い換えに弱い。意味が同じ言い換え（「バイク → オートバイ」など）でも、BLEU は元の 86%、CIDEr は 88% に下がり、文単位の BLEU では 70% の画像で元の候補のほうが高かった。CLIPScore は 101% で、元のほうが高いのは 48%（ほぼ半々）だった。言い換えに強い参照ありの指標には、評価指標と統計的検定の章の「BERTScore で意味的な近さを評価する」もある。
- CLIPScore は細かい誤りに鈍い。物の入れ替えは 92% の画像で見分けたが、位置・数の誤り（「の上に → の下に」「2人 → 1人」など）では値が元の 97% のままで、見分けたのは 70% だった。参照と画像の両方を使う RefCLIPScore は、物の入れ替えで 98%、位置・数で 80% と、CLIPScore より見分けた。
- 色の入れ替えでは、文単位の BLEU の 48% が同点だった。元の語（「赤い → 青い」の「赤い」）も新しい語（「青い」）も参照に出てこなければ、一致する n-gram は変わらないからである。CIDEr はコサイン類似度で、候補の n-gram のベクトルの長さで割るので、参照と一致しない語でも、元の語と新しい語の IDF が違えば値が変わる。そのため同点は 0% だったが、色を入れ替えたほうが高くなった画像も 17% あった（元のほうが高いのは 83%）。誤りを入れれば必ず下がるわけではない。
- CIDEr は、多くの画像の参照に出てくる n-gram を IDF で割り引く（「が」は 0.00、「男性」は 1.99、「パソコン」は 3.76）。どの画像にも当てはまりそうな「男性が立っている」を全画像に出すと、BLEU は人の値の約 23% (5.9) なのに、CIDEr は約 4〜5% (0.036) まで下がった。IDF は評価する画像の集合の参照から求めるので、集合が変われば値も変わる（上の表では、誤りの種類ごとに集合が違うので、元の候補との比で示した）。
- CLIPScore は参照なしで使える反面、値は 0〜2.5 のうち下のほうに集まる（人のキャプションで 0.899、コサイン類似度にして約 0.36）。また、ありがちな候補にも 0.413 が付き、別の画像のキャプション (0.250) より高かった。係数の 2.5 は元の論文（英語の CLIP）で使われた値で、モデルが変われば値の範囲も変わる。違うモデルの CLIPScore 同士は比べない。CLIP の仕組みは、この章の「CLIP で画像と日本語の文を同じ空間に埋め込み、検索・分類する」を参照。
- ここでは 300 枚（誤りの種類ごとに 39〜123 枚）の小さな実験で、置き換えも表に基づく機械的なもの。システムを比べるときは、評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」で信頼区間を求め、人手評価との相関は「自動評価指標と人手評価の相関を調べる」で確かめる。なお STAIR Captions の検証セットは、clip-japanese-base のモデルカードで検索の評価に使われているデータでもある。